# 01 · Análisis exploratorio de datos (EDA)
## Telco Customer Churn | Aprendizaje Automático

**Objetivo de negocio:** entender qué características describen a los clientes que abandonan la compañía (`Churn = Yes`) y evaluar si el dataset es adecuado para desarrollar una herramienta de priorización de retención.

**Alcance de este notebook:** carga, auditoría, limpieza justificada, balance de clases, análisis univariado y bivariado, prevención de *data leakage* y especificación de un preprocesamiento reproducible. **No se entrenan modelos, no se optimizan hiperparámetros y no se evalúa el conjunto de prueba en este notebook.**

**Fuente:** [Blastchar — Telco Customer Churn (Kaggle)](https://www.kaggle.com/datasets/blastchar/telco-customer-churn). Es un conjunto de datos de ejemplo difundido por IBM. Una fila representa a un cliente; `Churn` indica si abandonó el servicio. No se dispone de una secuencia temporal de capturas ni de resultados de campañas de retención.

### Protocolo metodológico

1. Auditar estructura, tipos, identificadores, nulos, duplicados e inconsistencias.
2. Aplicar únicamente correcciones deterministas y explicables que no aprendan estadísticas del dataset.
3. **Reservar el 20 % de prueba con partición estratificada y semilla fija.**
4. Hacer el EDA que relaciona predictores con `Churn` **solo sobre entrenamiento**.
5. Definir las transformaciones como `ColumnTransformer` + `Pipeline` **sin hacer `.fit()`** todavía.
6. Cerrar con una tabla de hallazgos, acciones y variables conservadas, generada con los datos realmente leídos.

> No confundir correlaciones o tasas por segmento con relaciones causales. El dataset sirve para una demostración de clasificación; no permite inferir por sí solo que una intervención vaya a evitar abandonos.

## 0. Entorno de ejecución

**Instalación sugerida (solo una vez, si faltan paquetes):**

```bash
pip install "kagglehub>=0.3" "pandas>=2.0" "numpy>=1.24" "matplotlib>=3.7" "seaborn>=0.12" "scikit-learn>=1.2" "ipython>=8" "jupyterlab>=4"
```

Si ejecutas en Google Colab, puedes usar `%pip install ...` en una celda. El notebook busca primero el CSV en `./data/` o en la carpeta actual; si no existe, lo descarga con la instrucción `kagglehub.dataset_download(...)` solicitada.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

SEED = 42
TAMANO_TEST = 0.20
ARCHIVO_CSV = "WA_Fn-UseC_-Telco-Customer-Churn.csv"

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 110)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (9, 5)

print("Pandas:", pd.__version__)
print("Semilla:", SEED)

## 1. Carga y procedencia

No se sustituye el dataset por datos sintéticos ni se lee una fuente diferente de manera silenciosa. El archivo cargado se registra para facilitar la reproducibilidad. Si Kaggle cambia la versión, los conteos del análisis se calculan de nuevo y no se dan por supuestos.

In [ ]:
rutas_locales = [Path("data") / ARCHIVO_CSV, Path(ARCHIVO_CSV)]
ruta_csv = next((r for r in rutas_locales if r.is_file()), None)

if ruta_csv is None:
    try:
        import kagglehub
    except ImportError as exc:
        raise ImportError(
            "No se encontró el CSV local y falta kagglehub. "
            "Instala kagglehub o coloca el archivo en ./data/."
        ) from exc
    path = kagglehub.dataset_download("blastchar/telco-customer-churn")
    candidatos = list(Path(path).rglob(ARCHIVO_CSV))
    if not candidatos:
        candidatos = list(Path(path).rglob("*.csv"))
    if len(candidatos) != 1:
        raise FileNotFoundError(
            f"Se esperaba un CSV único en {path}. Se encontraron: {candidatos}"
        )
    ruta_csv = candidatos[0]
else:
    path = str(ruta_csv.parent)

print("Path to dataset files:", path)
print("Archivo utilizado:", ruta_csv.resolve())

df_raw = pd.read_csv(ruta_csv, dtype={"customerID": "string"})
display(df_raw.head())
print(f"Dimensión original: {df_raw.shape[0]:,} filas × {df_raw.shape[1]} columnas")

## 2. Diccionario de variables y tipología

El tipo de Pandas **no siempre es el tipo estadístico**: `SeniorCitizen` es un indicador categórico codificado con 0/1, mientras que `TotalCharges` debería ser numérica aunque aparezca originalmente como texto. La variable objetivo tampoco debe introducirse entre las variables predictoras.

- **Identificador:** `customerID` (único por cliente; no es una variable explicativa).
- **Objetivo:** `Churn` (binaria: `No` → 0, `Yes` → 1).
- **Numéricas (3):** `tenure`, `MonthlyCharges`, `TotalCharges`.
- **Categóricas (16):** las variables de perfil, contratación, servicios y método de pago que se enumeran en la próxima celda; se incluye `SeniorCitizen`.

No se impone una jerarquía ordinal artificial a `Contract`, `PaymentMethod`, `InternetService` ni a las categorías de servicios. Las categorías «No internet service» y «No phone service» aportan contexto y **no son valores faltantes**.

In [ ]:
ID_COL = "customerID"
TARGET = "Churn"
NUMERICAS = ["tenure", "MonthlyCharges", "TotalCharges"]
CATEGORICAS = [
    "gender", "SeniorCitizen", "Partner", "Dependents", "PhoneService",
    "MultipleLines", "InternetService", "OnlineSecurity", "OnlineBackup",
    "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies",
    "Contract", "PaperlessBilling", "PaymentMethod"
]
PREDICTORAS = NUMERICAS + CATEGORICAS
ESPERADAS = set([ID_COL, TARGET] + PREDICTORAS)

if set(df_raw.columns) != ESPERADAS:
    raise ValueError(
        "El esquema cambió. Revisar columnas inesperadas / ausentes antes de seguir. "
        f"Ausentes: {sorted(ESPERADAS - set(df_raw.columns))}; "
        f"adicionales: {sorted(set(df_raw.columns) - ESPERADAS)}"
    )

diccionario = pd.DataFrame({
    "Variable": df_raw.columns,
    "Rol": ["Identificador" if c == ID_COL else
            "Objetivo" if c == TARGET else
            "Numérica" if c in NUMERICAS else "Categórica" for c in df_raw.columns],
    "Tipo original": [str(df_raw[c].dtype) for c in df_raw.columns],
    "Valores distintos": [df_raw[c].nunique(dropna=True) for c in df_raw.columns],
    "Nulos Pandas (sin limpieza)": [int(df_raw[c].isna().sum()) for c in df_raw.columns]
})
display(diccionario)
print(f"Predictoras: {len(PREDICTORAS)} = {len(NUMERICAS)} numéricas + {len(CATEGORICAS)} categóricas")

## 3. Auditoría de calidad de datos

### 3.1. Espacios vacíos, coerción de tipos y duplicados

Para contar correctamente los faltantes, primero se eliminan espacios de los textos y los campos en blanco se marcan como `NaN`. `TotalCharges` se convierte con `pd.to_numeric(errors='coerce')`, registrando cuántos valores no pudieron convertirse. Se validan también duplicados exactos y duplicados por ID.

**Regla de filas:** no eliminar clientes por tener `TotalCharges` vacío. En este dataset, esos casos suelen tener `tenure = 0`: interpretamos que no existe todavía un importe histórico facturado y proponemos `TotalCharges = 0` **solo en ese supuesto**. Si aparecieran valores faltantes con `tenure > 0`, no se rellenarían con una media calculada sobre todo el dataset; quedarían para imputación dentro del pipeline ajustado en entrenamiento.

In [ ]:
df = df_raw.copy(deep=True)

# Limpiar espacios en todas las columnas de tipo texto, sin convertir NaN a la cadena "nan".
for columna in df.select_dtypes(include=["object", "string"]).columns:
    df[columna] = df[columna].astype("string").str.strip()
    df[columna] = df[columna].replace("", pd.NA)

# Inspección antes de convertir: los espacios vacíos de TotalCharges ya son NaN.
blancos_totalcharges = int(df["TotalCharges"].isna().sum())
valores_originales_no_numericos = df["TotalCharges"].dropna()
valores_no_convertibles = int(pd.to_numeric(valores_originales_no_numericos, errors="coerce").isna().sum())

for columna in NUMERICAS:
    df[columna] = pd.to_numeric(df[columna], errors="coerce")

# Mantener SeniorCitizen como categoría semántica, no como magnitud continua.
valores_senior = set(df["SeniorCitizen"].dropna().astype("string").unique())
if not valores_senior.issubset({"0", "1"}):
    raise ValueError(f"SeniorCitizen tiene códigos inesperados: {valores_senior}")
df["SeniorCitizen"] = df["SeniorCitizen"].astype("string").map({"0": "No", "1": "Yes"})

# El target se convierte de forma explícita. Nunca se infiere a partir de LabelEncoder.
valores_churn = set(df[TARGET].dropna().unique())
if valores_churn != {"No", "Yes"} or df[TARGET].isna().any():
    raise ValueError(f"Etiquetas de Churn no válidas o incompletas: {valores_churn}")
df[TARGET] = df[TARGET].map({"No": 0, "Yes": 1}).astype("int8")

faltantes_antes = df.isna().sum().sort_values(ascending=False)
ids_duplicados = int(df[ID_COL].duplicated().sum())
filas_duplicadas = int(df.duplicated().sum())

print(f"Blancos originales en TotalCharges: {blancos_totalcharges}")
print(f"Textos no convertibles adicionales en TotalCharges: {valores_no_convertibles}")
print(f"IDs duplicados: {ids_duplicados}")
print(f"Filas completas duplicadas: {filas_duplicadas}")
display(faltantes_antes.rename("Nulos tras conversión").to_frame())

if ids_duplicados or filas_duplicadas:
    raise ValueError(
        "Se detectaron duplicados. Antes de dividir se debe revisar si son "
        "registros repetidos o múltiples capturas del mismo cliente; "
        "no se eliminan automáticamente."
    )

In [ ]:
# Validar evidencia antes de decidir si la imputación de cero es apropiada.
faltan_total = df["TotalCharges"].isna()
mask_cero_justificado = faltan_total & df["tenure"].eq(0)
mask_sin_justificacion = faltan_total & ~df["tenure"].eq(0)

print("TotalCharges faltante y tenure=0:", int(mask_cero_justificado.sum()))
print("TotalCharges faltante y tenure!=0 (o desconocido):", int(mask_sin_justificacion.sum()))
display(df.loc[faltan_total, [ID_COL, "tenure", "MonthlyCharges", "TotalCharges", TARGET]].head(15))

# Decisión de dominio: no hay facturación histórica acumulada cuando la antigüedad es cero.
n_ceros_imputados = int(mask_cero_justificado.sum())
df.loc[mask_cero_justificado, "TotalCharges"] = 0.0

faltantes_despues = df.isna().sum().sort_values(ascending=False)
display(pd.DataFrame({
    "Antes de corrección": faltantes_antes,
    "Después de corrección": faltantes_despues
}).query('`Antes de corrección` > 0 or `Después de corrección` > 0'))
print("Filas eliminadas por faltantes:", 0)

### 3.2. Validaciones de coherencia

Se comprueban reglas lógicas entre servicios y la plausibilidad básica de importes y antigüedad. Los valores altos de gasto no se eliminan automáticamente como «outliers»: un cliente con permanencia larga puede tener un `TotalCharges` alto legítimo, y los clientes con facturas elevadas pueden ser precisamente relevantes para el negocio.

**Regla:** se reportan anomalías; si existen, se revisan antes de modelar. No se corrigen inventando servicios o importes.

In [ ]:
servicios_internet = [
    "OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport",
    "StreamingTV", "StreamingMovies"
]

mask_phone_no_incoherente = df["PhoneService"].eq("No") & ~df["MultipleLines"].eq("No phone service")
mask_phone_si_incoherente = df["PhoneService"].eq("Yes") & df["MultipleLines"].eq("No phone service")
mask_internet_no_incoherente = df["InternetService"].eq("No") & (
    df[servicios_internet].ne("No internet service").any(axis=1)
)
mask_internet_si_incoherente = df["InternetService"].ne("No") & (
    df[servicios_internet].eq("No internet service").any(axis=1)
)

controles = {
    "tenure negativo": int(df["tenure"].lt(0).sum()),
    "MonthlyCharges negativo": int(df["MonthlyCharges"].lt(0).sum()),
    "TotalCharges negativo": int(df["TotalCharges"].lt(0).sum()),
    "Sin teléfono pero MultipleLines incoherente": int(mask_phone_no_incoherente.sum()),
    "Con teléfono pero MultipleLines='No phone service'": int(mask_phone_si_incoherente.sum()),
    "Sin internet pero servicios incoherentes": int(mask_internet_no_incoherente.sum()),
    "Con internet pero 'No internet service'": int(mask_internet_si_incoherente.sum()),
}
control_df = pd.Series(controles, name="Casos detectados").to_frame()
display(control_df)
print("Anomalías lógicas detectadas:", sum(controles.values()))

if sum(controles.values()) > 0:
    warnings.warn("Hay inconsistencias que deben revisarse antes de entrenar. No se corrigieron automáticamente.")

## 4. Análisis de la variable objetivo y balance

La variable positiva es **`Churn = 1` (abandono)**. Un dataset no necesita tener exactamente 50/50 para ser útil, pero una mayoría clara de una clase vuelve engañosa la sola *Accuracy*. Se informa cantidad, porcentaje, relación mayoritaria/minoritaria y el rendimiento trivial que lograría un clasificador que siempre predijera «No abandona».

**Criterio descriptivo de este notebook:** denominaremos «desbalance moderado» a la distribución aproximada 3:1 de este dataset. No es un umbral universal y la decisión de reponderar clases será parte de la validación cruzada posterior.

In [ ]:
conteos_churn = df[TARGET].value_counts().reindex([0, 1], fill_value=0)
porcentajes_churn = 100 * conteos_churn / len(df)
balance_df = pd.DataFrame({
    "Clase": ["No abandona (0)", "Abandona (1)"],
    "Clientes": conteos_churn.values,
    "Porcentaje (%)": porcentajes_churn.values.round(2)
})
display(balance_df)

n_mayoritaria, n_minoritaria = int(conteos_churn.max()), int(conteos_churn.min())
relacion_clases = n_mayoritaria / n_minoritaria if n_minoritaria else np.inf
accuracy_trivial = n_mayoritaria / len(df)
print(f"Relación clase mayoritaria / minoritaria: {relacion_clases:.2f}:1")
print(f"Accuracy del clasificador de mayoría: {accuracy_trivial:.2%}")
print("¿Equilibrio aproximado 50/50?:", "Sí" if abs(porcentajes_churn[0] - porcentajes_churn[1]) < 10 else "No")

fig, ax = plt.subplots(figsize=(7, 4.5))
barras = ax.bar(balance_df["Clase"], balance_df["Clientes"], color=["#426e91", "#e28f67"])
ax.bar_label(barras, labels=[f"{n:,}  ({p:.1f} %)" for n, p in zip(balance_df["Clientes"], balance_df["Porcentaje (%)"])], padding=4)
ax.set(title="Distribución de Churn · dataset completo (auditoría inicial)", ylabel="Número de clientes", ylim=(0, n_mayoritaria*1.15))
plt.tight_layout()
plt.show()

## 5. Separación estratificada y aislamiento del test

**Punto crítico del proyecto:** la partición se realiza **antes de analizar la relación de las características con el abandono**. Usamos `stratify=df['Churn']`, `test_size=0.20` y `random_state=42`. Todos los gráficos de asociaciones a partir de este punto usan exclusivamente `df_train_eda`.

- No se ajustan imputadores, escaladores ni codificadores usando el test.
- No se eligen variables ni hiperparámetros observando el test.
- El `test` queda reservado para **una evaluación final** al terminar el entrenamiento, la optimización y la selección de modelo en notebooks posteriores.
- En las búsquedas futuras, cada `fit` de preprocesamiento debe ocurrir *dentro* de cada fold de validación cruzada.

> La comprobación de la distribución global de `Churn` y de problemas de calidad es una auditoría descriptiva. No se usan las relaciones entre variables y objetivo del conjunto de prueba para decidir el modelo.

In [ ]:
idx_train, idx_test = train_test_split(
    df.index.to_numpy(),
    test_size=TAMANO_TEST,
    random_state=SEED,
    stratify=df[TARGET]
)

# Copias independientes, con índices originales para trazabilidad.
df_train_eda = df.loc[idx_train].copy()
df_test_reservado = df.loc[idx_test].copy()

X_train = df_train_eda[PREDICTORAS].copy()
y_train = df_train_eda[TARGET].copy()
X_test = df_test_reservado[PREDICTORAS].copy()
y_test = df_test_reservado[TARGET].copy()

assert set(X_train.index).isdisjoint(X_test.index), "¡Fuga: índices solapados!"
assert ID_COL not in X_train.columns and TARGET not in X_train.columns
assert len(df_train_eda) + len(df_test_reservado) == len(df)

print(f"Conjunto de entrenamiento / EDA: {len(X_train):,} filas, {X_train.shape[1]} predictoras")
print(f"Conjunto de prueba reservado: {len(X_test):,} filas (sin exploración del objetivo)")
print(f"Tasa Churn en entrenamiento: {y_train.mean():.2%}")

## 6. EDA univariado: variables numéricas

Las tres numéricas no representan lo mismo: `tenure` mide meses de antigüedad, `MonthlyCharges` gasto mensual y `TotalCharges` cargos históricos acumulados. Por ello tienen escalas diferentes y podrían mostrar asimetría y valores extremos legítimos. La estandarización para regresión logística será una decisión del pipeline, **no una transformación calculada aquí con todo el dataset**.

In [ ]:
resumen_numericas = X_train[NUMERICAS].describe().T
resumen_numericas["Nulos (train)"] = X_train[NUMERICAS].isna().sum()
resumen_numericas["Asimetría"] = X_train[NUMERICAS].skew()
display(resumen_numericas.round(2))

fig, axs = plt.subplots(1, 3, figsize=(16, 4))
for ax, columna in zip(axs, NUMERICAS):
    sns.histplot(data=df_train_eda, x=columna, kde=True, ax=ax, color="#4477a6", bins=30)
    ax.set_title(columna)
    ax.set_ylabel("Clientes")
fig.suptitle("Distribuciones numéricas · solo entrenamiento", y=1.04, fontsize=14)
plt.tight_layout()
plt.show()

## 7. EDA univariado: variables categóricas

Se estudia el **número de categorías** de cada variable, posibles categorías raras y su proporción. Esto respalda la selección de `OneHotEncoder`, compatible con múltiples niveles nominales y con categorías desconocidas en inferencia. Una codificación numérica arbitraria (`0,1,2,...`) introduciría un orden no necesariamente existente.

In [ ]:
resumen_categoricas = pd.DataFrame({
    "Variable": CATEGORICAS,
    "Niveles (train)": [X_train[c].nunique(dropna=True) for c in CATEGORICAS],
    "Nulos (train)": [int(X_train[c].isna().sum()) for c in CATEGORICAS],
    "Categorías observadas (train)": [", ".join(sorted(X_train[c].dropna().astype(str).unique())) for c in CATEGORICAS],
})
display(resumen_categoricas)

fig, axs = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, col in zip(axs, ["Contract", "InternetService"]):
    orden = X_train[col].value_counts().index
    sns.countplot(data=df_train_eda, x=col, order=orden, ax=ax, color="#4477a6")
    ax.set(title=f"Frecuencia de {col}", xlabel=col, ylabel="Clientes")
    ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

## 8. EDA bivariado: categorías y tasa de abandono

Comparar *conteos* únicamente puede confundir tamaño del segmento con riesgo. Para cada categoría calculamos la **tasa de churn dentro del segmento** (`clientes que abandonan / clientes del segmento`). Se muestra también el tamaño de cada grupo para no sobreinterpretar categorías pequeñas.

Se destacan `Contract`, `InternetService`, `PaymentMethod` y `TechSupport` por su relevancia comercial y variedad de niveles. Estas relaciones son descriptivas y no demuestran causalidad; por ejemplo, el tipo de contrato puede asociarse a la antigüedad y a otros factores.

In [ ]:
def tasa_por_segmento(datos, variable):
    resultado = (
        datos.groupby(variable, dropna=False)[TARGET]
        .agg(clientes="size", abandonos="sum", tasa_churn="mean")
        .reset_index()
        .sort_values("tasa_churn", ascending=False)
    )
    resultado["tasa_churn_pct"] = 100 * resultado["tasa_churn"]
    return resultado

variables_segmentos = ["Contract", "InternetService", "PaymentMethod", "TechSupport"]
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
for ax, variable in zip(axes.ravel(), variables_segmentos):
    tabla = tasa_por_segmento(df_train_eda, variable)
    sns.barplot(data=tabla, y=variable, x="tasa_churn_pct", ax=ax, color="#d68362", errorbar=None)
    ax.set(xlabel="Tasa de abandono (%)", ylabel="", title=f"Churn según {variable}", xlim=(0, 100))
    for i, fila in enumerate(tabla.itertuples(index=False)):
        ax.text(float(fila.tasa_churn_pct) + 1, i, f"n={int(fila.clientes)}", va="center", fontsize=9)
    ax.set_xlim(0, max(55, float(tabla["tasa_churn_pct"].max()) + 20))
plt.suptitle("Tasa de Churn por segmento · solo entrenamiento", y=1.01, fontsize=15)
plt.tight_layout()
plt.show()

for variable in variables_segmentos:
    print(f"\n{variable}")
    display(tasa_por_segmento(df_train_eda, variable)[[variable, "clientes", "abandonos", "tasa_churn_pct"]].round(2))

## 9. EDA bivariado: numéricas y abandono

Se comparan distribuciones por clase sin usar la partición de prueba. La asociación entre baja permanencia y churn (si aparece) puede indicar un segmento prioritario para campañas; diferencias de cargos pueden reflejar mezcla de planes. No se fijará ningún umbral manual de clasificación con estos gráficos.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, columna in zip(axes, NUMERICAS):
    sns.boxplot(
        data=df_train_eda, x=TARGET, y=columna,
        order=[0, 1], ax=ax, color="#78a6c2", showfliers=False
    )
    ax.set(xlabel="Churn (0 = No; 1 = Sí)", title=columna)
fig.suptitle("Distribución de variables numéricas según Churn · entrenamiento", y=1.03, fontsize=14)
plt.tight_layout()
plt.show()

comparacion_numericas = df_train_eda.groupby(TARGET)[NUMERICAS].agg(["median", "mean"])
display(comparacion_numericas.round(2))

## 10. Correlación, redundancia y lectura comercial

La correlación de Pearson entre numéricas se utiliza como diagnóstico, **no como motivo automático para quitar una variable**. `TotalCharges` acumula montos a lo largo de la permanencia y puede ser redundante con `tenure` y `MonthlyCharges`; aun así, conviene mantenerla inicialmente y evaluar su aporte con validación cruzada.

**Posible riesgo temporal:** antes de desplegar el modelo hay que confirmar que `TotalCharges`, `tenure` y las demás variables se conocen **en el instante de la predicción**. Si alguna estuviera calculada después del abandono, habría fuga temporal de información. Este dataset de ejemplo no proporciona fechas suficientes para demostrarlo.

In [ ]:
corr = df_train_eda[NUMERICAS + [TARGET]].corr(method="pearson")
display(corr.round(3))
plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, square=True)
plt.title("Correlación (Pearson) · solo entrenamiento")
plt.tight_layout()
plt.show()

## 11. Segmentación exploratoria para una futura propuesta de negocio

Una hipótesis comercial razonable es que **permanencia y nivel de factura mensual** pueden servir para priorizar una oferta de retención. Se explora una cuadrícula de segmentos, usando bandas analíticas del entrenamiento. No se incluyen estas columnas derivadas en el pipeline inicial: crear umbrales basados en cuantiles o métricas del dataset completo podría introducir decisiones difíciles de validar.

**Importante:** el análisis de segmentos solo muestra asociación; no estima ahorro económico, uplift causal ni efectividad real de campañas.

In [ ]:
segmentado = df_train_eda[["tenure", "MonthlyCharges", TARGET]].copy()

# Bandas de antigüedad con cortes fijos interpretables, sin aprender del test.
segmentado["Banda_antiguedad"] = pd.cut(
    segmentado["tenure"], bins=[-1, 12, 24, 48, np.inf],
    labels=["0–12", "13–24", "25–48", "49+"], include_lowest=True
)

# Bandas descriptivas de mensualidad: usan cuantiles del TRAIN, no del test.
# Si los cuantiles coinciden, qcut elimina cortes redundantes de forma explícita.
segmentado["Banda_cargo_mensual"] = pd.qcut(
    segmentado["MonthlyCharges"], q=3, labels=False, duplicates="drop"
).map({0: "Bajo", 1: "Medio", 2: "Alto"})

pivot_riesgo = segmentado.pivot_table(
    index="Banda_antiguedad", columns="Banda_cargo_mensual",
    values=TARGET, aggfunc="mean", observed=True
)
pivot_tamanio = segmentado.pivot_table(
    index="Banda_antiguedad", columns="Banda_cargo_mensual",
    values=TARGET, aggfunc="size", observed=True
)

print("Tasa de Churn por segmento (entrenamiento):")
display((100 * pivot_riesgo).round(1).add_suffix(" %"))
print("Tamaño por segmento (entrenamiento):")
display(pivot_tamanio)

plt.figure(figsize=(8, 5))
sns.heatmap(100 * pivot_riesgo, annot=True, fmt=".1f", cmap="YlOrRd", cbar_kws={"label": "Churn (%)"})
plt.title("Abandono por antigüedad y gasto mensual · entrenamiento")
plt.xlabel("Nivel de cargo mensual (terciles TRAIN)")
plt.ylabel("Antigüedad en meses")
plt.tight_layout()
plt.show()

## 12. Decisión de preprocesamiento para la siguiente fase

| Grupo | Acción propuesta | Justificación |
|---|---|---|
| `customerID` | **Excluir de `X`**, conservar en la auditoría | Identificador sin semántica predictiva; puede favorecer memorización. |
| `Churn` | Convertir `Yes` → 1 y `No` → 0; separar como `y` | Clasificación binaria, clase positiva = abandono. |
| `tenure`, `MonthlyCharges`, `TotalCharges` | `SimpleImputer(strategy='median')` + `StandardScaler()` dentro del pipeline | Robustez ante faltantes futuros y escalas diferentes; útil para regresión logística y SVM. `TotalCharges` ya recibió la corrección determinista de `tenure=0`. |
| 16 categóricas (incluye `SeniorCitizen`) | `SimpleImputer(strategy='most_frequent')` + `OneHotEncoder(handle_unknown='ignore')` | Evita orden arbitrario, conserva «No internet service» como categoría, acepta categorías nuevas al desplegar. |
| Filas con `TotalCharges` faltante y `tenure=0` | **Conservar** e imputar `TotalCharges=0` | Regla de negocio explícita, aplicada fila a fila y sin estadísticas globales. |
| Outliers numéricos | No eliminar de manera automática | Los gastos grandes y antigüedades largas pueden ser casos válidos. |
| Desbalance de `Churn` | Estratificar; evaluar `class_weight` en CV | Evita considerar como buen modelo a uno que siempre prediga «No». No aplicar SMOTE antes del split. |

**Por qué OHE y no `LabelEncoder`/codificación ordinal:** `Contract` y `PaymentMethod` no son niveles de una magnitud continua; una codificación 0–1–2–3 introduciría distancias inexistentes. Con aproximadamente dieciséis variables nominales de baja cardinalidad, OHE es viable. `SeniorCitizen` se expresa como «Yes/No» por consistencia semántica. Se conserva el conjunto completo de *dummies* para facilitar trazabilidad hacia SHAP y LIME; una regresión logística regularizada maneja la redundancia resultante.

**Atención:** el escalado es importante en modelos sensibles a magnitudes (Logistic Regression, SVM). Random Forest no lo necesita, pero se puede usar un preprocesamiento común para una comparación controlada. El uso de `OneHotEncoder(sparse_output=False)` es asequible para este dataset pequeño; en uno grande convendría salida dispersa.

In [ ]:
# ESPECIFICACIÓN, SIN FIT: no se aprende nada de los folds ni del holdout.
pipeline_numerico = Pipeline(steps=[
    ("imputador", SimpleImputer(strategy="median")),
    ("escalador", StandardScaler())
])

pipeline_categorico = Pipeline(steps=[
    ("imputador", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocesador = ColumnTransformer(transformers=[
    ("num", pipeline_numerico, NUMERICAS),
    ("cat", pipeline_categorico, CATEGORICAS)
], remainder="drop", verbose_feature_names_out=True)

print("ColumnTransformer definido, NO ajustado.")
print("Numéricas:", NUMERICAS)
print("Categóricas:", CATEGORICAS)
print("Para la fase 2: Pipeline([('prep', preprocesador), ('modelo', clasificador)])")
print("Usar StratifiedKFold y búsqueda de hiperparámetros SOLO sobre X_train / y_train.")

## 13. Conclusiones del EDA (generadas con los datos de ejecución)

La siguiente celda crea la **tabla final de diagnóstico y decisiones**. Distingue entre columnas originales y predictoras, conserva el recuento real de nulos corregidos, registra las filas descartadas y documenta cómo se tratarán las variables. Los conteos se calculan dinámicamente: no dependen de copiar cifras de una solución externa.

In [ ]:
# Resumen de relaciones observadas SOLO en entrenamiento.
tasas_contrato = tasa_por_segmento(df_train_eda, "Contract")
segmento_contrato_mayor = tasas_contrato.iloc[0]
segmento_contrato_menor = tasas_contrato.iloc[-1]

faltantes_pendientes = int(df[PREDICTORAS].isna().sum().sum())
num_fil_originales = len(df_raw)
num_fil_finales = len(df)
num_cols_originales = df_raw.shape[1]
num_cols_finales = len(PREDICTORAS) + 1

resumen_decisiones = pd.DataFrame([
    ("Fuente", "blastchar/telco-customer-churn", "Archivo Kaggle identificado; carga reproducible"),
    ("Dimensión original", f"{num_fil_originales:,} filas × {num_cols_originales} columnas", "Auditar estructura antes de transformar"),
    ("Variable objetivo", "Churn: 0 = No, 1 = Yes", "Clasificación binaria; abandono es clase positiva"),
    ("Distribución de Churn", f"No: {conteos_churn[0]:,} ({porcentajes_churn[0]:.2f} %); Sí: {conteos_churn[1]:,} ({porcentajes_churn[1]:.2f} %)", "Desbalance de clases: preservar mediante estratificación"),
    ("Relación de clases", f"{relacion_clases:.2f}:1 (mayoritaria/minoritaria)", "Accuracy sola sería insuficiente; priorizar Recall, F1, ROC-AUC y Precision"),
    ("Baseline de mayoría", f"Accuracy = {accuracy_trivial:.2%}", "Una mejora debe superar la regla trivial sin ignorar la clase 1"),
    ("Identificador", ID_COL, "Se mantiene para trazabilidad; se EXCLUYE de X"),
    ("Variables numéricas", f"{len(NUMERICAS)}: {', '.join(NUMERICAS)}", "Imputación mediana y escalado aprendidos solo en train/folds"),
    ("Variables categóricas", f"{len(CATEGORICAS)} (incluye SeniorCitizen)", "Imputación modal y OneHotEncoder con handle_unknown='ignore'"),
    ("TotalCharges blanco", f"{blancos_totalcharges} espacios vacíos; {valores_no_convertibles} errores adicionales", "Se convirtió a numérico y se diagnosticaron faltantes"),
    ("Ceros de TotalCharges imputados", f"{n_ceros_imputados} casos con tenure=0", "Corrección de dominio específica, sin eliminar clientes"),
    ("Faltantes de predictores restantes", str(faltantes_pendientes), "Si existieran, se tratarían DENTRO del pipeline"),
    ("Duplicados", f"{ids_duplicados} IDs; {filas_duplicadas} filas completas", "No se encontraron duplicados por resolver si ambos son 0"),
    ("Inconsistencias de servicio", f"{sum(controles.values())} incidencias de los controles aplicados", "No modificar automáticamente; revisar si aparecen"),
    ("Filas eliminadas", str(num_fil_originales - num_fil_finales), "No descartar filas por TotalCharges vacío cuando tenure=0"),
    ("Columnas excluidas del modelo", ID_COL, "No usar identificadores ni Churn como predictores"),
    ("Datos preparados", f"{num_fil_finales:,} filas × {num_cols_finales} columnas (19 X + 1 y)", "Conservar todas las variables predictoras iniciales"),
    ("Partición protegida", f"Train: {len(X_train):,} | Test: {len(X_test):,} (80/20 estratificado)", "Test reservado hasta evaluación final del modelo"),
    ("Mayor tasa de churn por contrato (train)", f"{segmento_contrato_mayor['Contract']}: {segmento_contrato_mayor['tasa_churn_pct']:.1f} % (n={int(segmento_contrato_mayor['clientes'])})", "Hipótesis de segmentación, NO causalidad"),
    ("Menor tasa de churn por contrato (train)", f"{segmento_contrato_menor['Contract']}: {segmento_contrato_menor['tasa_churn_pct']:.1f} % (n={int(segmento_contrato_menor['clientes'])})", "Verificar estabilidad con futuras validaciones"),
    ("Outliers", "Ninguna eliminación automática", "Importes y antigüedades altos pueden ser reales"),
    ("Estado del preprocesador", "Definido, no entrenado", "Ajustar el pipeline dentro de cada fold de validación cruzada"),
    ("Riesgo de leakage", "Temporalidad de variables no verificable en dataset de ejemplo", "Validar disponibilidad de variables al predecir y NO analizar test"),
], columns=["Aspecto", "Resultado observado", "Decisión y justificación"])

display(Markdown("### Tabla resumen final del EDA"))
display(resumen_decisiones)

display(Markdown("### Variables con las que continuaremos"))
variables_finales = pd.DataFrame([
    {"Variable": c,
     "Tipo para modelado": "Numérica" if c in NUMERICAS else "Categórica",
     "Nulos pendientes en dataset limpio": int(df[c].isna().sum()),
     "Conservada": "Sí",
     "Tratamiento previsto": "Mediana + StandardScaler" if c in NUMERICAS else "Moda + OneHotEncoder"}
    for c in PREDICTORAS
])
display(variables_finales)

display(Markdown(
    f"**Resultado:** {len(PREDICTORAS)} predictores ({len(NUMERICAS)} numéricos y "
    f"{len(CATEGORICAS)} categóricos), {len(df):,} clientes, "
    f"{len(X_train):,} para desarrollo y {len(X_test):,} reservados para prueba. "
    "No se ha entrenado ni ajustado ningún modelo o transformador."
))

## 14. Puente hacia el proyecto completo (próximos notebooks)

**No ejecutar estas etapas sobre `X_test` todavía.** Este EDA deja disponibles `X_train`, `y_train`, `X_test`, `y_test` y `preprocesador` (no ajustado).

| Etapa posterior | Protocolo recomendado |
|---|---|
| Modelos base | `Pipeline(preprocesador + LogisticRegression)` vs `Pipeline(preprocesador + RandomForestClassifier)` con misma partición y mismas métricas; valorar `class_weight`. |
| Métricas | Accuracy, Precision, **Recall** (clientes en riesgo detectados), F1, ROC-AUC, matriz de confusión. Opcionalmente PR-AUC y curvas precision-recall por desbalance. |
| Optimización | Grid Search, Random Search y TPE de Optuna, todos dentro de entrenamiento y con **el mismo StratifiedKFold**, semilla y presupuestos comparables. |
| Multiobjetivo | Maximizar F1 de CV y minimizar latencia de inferencia medida en condiciones homogéneas; identificar no dominadas y graficar Pareto. No optimizar sobre test. |
| SHAP / LIME | Explicar la **clase 1 = abandono** usando el pipeline completo y las mismas filas. Agregar importancias de categorías OHE si se presenta una variable original. |
| Streamlit | 3 visualizaciones interactivas EDA, KPIs, métricas y Pareto; cargar artefactos `.joblib` ya entrenados; formulario con el mismo esquema de 19 predictoras. |
| Negocio | Ejemplo de hipótesis: priorización de retención. Validar con costos/beneficios explícitamente hipotéticos y, para adopción real, con experimentos de campañas. |

### Advertencias clave para defender ante el profesor

- El dataset es **una muestra de ejemplo**, no un registro operativo completo ni un experimento causal.
- Una tasa de churn por segmento no es una «probabilidad causal»; SHAP y LIME explican al **modelo**, no causas del abandono.
- No hay garantía de que una métrica alta en este dataset se reproduzca con datos nuevos de otra operadora (posible *dataset shift*).
- Si se ajustara `OneHotEncoder`, `StandardScaler`, imputadores, SMOTE o técnicas de selección sobre toda la base **antes de CV**, se produciría fuga de información.
- El conjunto de prueba debe usarse **una sola vez al final**, tras cerrar la selección de la estrategia y los hiperparámetros. Si posteriormente se ajusta el umbral, debe hacerse en entrenamiento/validación, nunca buscando el mejor umbral sobre test.

**Conclusión metodológica:** mantenemos inicialmente los 19 predictores con tratamiento específico según su naturaleza, corregimos faltantes con una regla de negocio verificable y protegemos el test. Esto crea una base defendible para optimización, XAI y el *business pitch* sin atribuir causalidad a los hallazgos del EDA.